In [1]:
import os
import sys
import json
from datetime import datetime
from pathlib import Path

import random
import numpy as np
import wandb
import torch
from torch_geometric.loader import DataLoader

base_dir = Path('/mnt/e/fyassine/+ad-early-detection')
sys.path.insert(0, str(base_dir))

import config

from model.DenseGAAE.models import DenseGraphAttentionAutoencoderConditioned
from model.DenseGAAE.dataset import GraphDatasetInMemoryFilteredDense
from model.DenseGAAE.train import train_model_with_val_notebook_train_loss


In [2]:
os.environ['WANDB_API_KEY'] = '24f49a72509540b42cb808a2a16ee55aedf93a65'
try:
    wandb.login()
except Exception:
    pass

WANDB_PROJECT = 'ad-early-detection'

wandb: Currently logged in as: lakhalfrajyassine (lakhalfrajyassine-technical-university-of-munich) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [3]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cuda


In [4]:
hyperparams_path = base_dir / 'configs' / 'gaae_dancer.json'
with open(hyperparams_path, 'r') as handle:
    hyperparams = json.load(handle)

seed = hyperparams['seed']
batch_size = hyperparams['batch_size']
learning_rate = hyperparams['learning_rate']
adj_loss_weight = hyperparams['adj_loss_weight']
n_epochs = hyperparams['epochs']
early_stopping_patience = hyperparams['early_stopping_patience']

out_features = hyperparams['latent_dim']
num_heads = hyperparams['num_heads']
cond_dim = hyperparams['cond_dim']
dropout = hyperparams['dropout']
model_save_path = hyperparams.get('model_save_path')

num_workers = hyperparams['num_workers']
file_variant = hyperparams.get('file_variant', 'z_transformed')

In [5]:
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
np.random.seed(seed)
random.seed(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

os.environ['PYTHONHASHSEED'] = str(seed)

def worker_init_fn(worker_id):
    worker_seed = seed + worker_id
    np.random.seed(worker_seed)
    random.seed(worker_seed)

In [6]:
dancer_train_dataset = GraphDatasetInMemoryFilteredDense(
    root=str(config.DANCER_RAW_DIR),
    filter_csv_path=str(config.DANCER_TRAIN_SPLIT_CSV),
    patient_info_path=str(config.DANCER_PATIENT_INFO_CSV),
    separator=';',
    file_variant=file_variant,
)

dancer_val_dataset = GraphDatasetInMemoryFilteredDense(
    root=str(config.DANCER_RAW_DIR),
    filter_csv_path=str(config.DANCER_VAL_SPLIT_CSV),
    patient_info_path=str(config.DANCER_PATIENT_INFO_CSV),
    separator=';',
    file_variant=file_variant,
)


print('Dataset sizes:')
print(f'  Train: {len(dancer_train_dataset)}')
print(f'  Val: {len(dancer_val_dataset)}')

Processing...
/mnt/e/fyassine/+ad-early-detection/model/DenseGAAE/dataset.py:66: RuntimeWarning: overflow encountered in cast
  corr_matrix = corr_matrix.astype(np.float32)
Done!
Processing...
/mnt/e/fyassine/+ad-early-detection/model/DenseGAAE/dataset.py:66: RuntimeWarning: overflow encountered in cast
  corr_matrix = corr_matrix.astype(np.float32)
Done!


Dataset sizes:
  Train: 112
  Val: 37


In [7]:
dataset_info = {
    'dataset_name': 'DANCER Healthy (Dense Graph)',
    'num_features': dancer_train_dataset[0].x.size(1),
    'train_dataset_size': len(dancer_train_dataset),
    'val_dataset_size': len(dancer_val_dataset),
    'dataset_split_seed': seed,
    'batch_size': batch_size
}

train_loader = DataLoader(
    dancer_train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    worker_init_fn=worker_init_fn,
    pin_memory=torch.cuda.is_available(),
)
val_loader = DataLoader(
    dancer_val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    worker_init_fn=worker_init_fn,
    pin_memory=torch.cuda.is_available(),
)

In [8]:
in_features = dancer_train_dataset[0].x.size(1)
hidden_dim = in_features

model = DenseGraphAttentionAutoencoderConditioned(
    in_features=in_features,
    hidden_dim=hidden_dim,
    out_features=out_features,
    cond_dim=cond_dim,
    num_heads=num_heads,
    dropout=dropout
).to(device)

model_config = {
    'model_type': model.__class__.__name__,
    'in_features': in_features,
    'hidden_size': hidden_dim,
    'latent_dim': out_features,
    'attention_heads': num_heads,
    'device': device.type,
    'dropout': dropout,
    'graph_mode': 'dense'
}

In [9]:
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [10]:
best_model, history = train_model_with_val_notebook_train_loss(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    device=device,
    batch_size=batch_size,
    learning_rate=learning_rate,
    model_config=model_config,
    adj_loss_weight=adj_loss_weight,
    epochs=n_epochs,
    early_stopping_patience=early_stopping_patience,
    dataset_info=dataset_info,
    project_name=WANDB_PROJECT
)

run_timestamp = datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
wandb_run_name = wandb.run.name if wandb.run else f'run_{run_timestamp}'
run_artifact_dir = os.path.join('checkpoints', str(wandb_run_name))
os.makedirs(run_artifact_dir, exist_ok=True)

model_filename = f'model_{wandb_run_name}.pth'
model_file = os.path.join(run_artifact_dir, model_filename)
torch.save(best_model, model_file)
print(f'Saved best model to {model_file}')

config_to_save = {
    'run_name': wandb_run_name,
    'timestamp': run_timestamp,
    'dataset_info': dataset_info,
    'model_config': model_config,
    'training_config': {
        'batch_size': batch_size,
        'learning_rate': learning_rate,
        'adj_loss_weight': adj_loss_weight,
        'epochs': n_epochs,
        'early_stopping_patience': early_stopping_patience
    }
}

def json_serial(obj):
    if isinstance(obj, (datetime, torch.device)):
        return str(obj)
    raise TypeError(f'Type {type(obj)} not serializable')

config_filename = 'run_config.json'
config_file = os.path.join(run_artifact_dir, config_filename)
with open(config_file, 'w') as handle:
    json.dump(config_to_save, handle, indent=4, default=json_serial)
print(f'Saved run configuration to {config_file}')

Training Progress:   0%|          | 0/1000 [00:00<?, ?it/s]

Exception ignored in: <function _releaseLock at 0x7cd00abecb80>
Traceback (most recent call last):
  File "/mnt/e/smoteval/conda_envs/early_ad/lib/python3.12/logging/__init__.py", line 243, in _releaseLock
    def _releaseLock():
    
KeyboardInterrupt: 


RuntimeError: DataLoader worker (pid(s) 828046) exited unexpectedly

Error in callback <bound method _WandbInit._post_run_cell_hook of <wandb.sdk.wandb_init._WandbInit object at 0x7cce765e6990>> (for post_run_cell), with arguments args (<ExecutionResult object at 7cce4a209f10, execution_count=10 error_before_exec=None error_in_exec=DataLoader worker (pid(s) 828046) exited unexpectedly info=<ExecutionInfo object at 7cce4a20a3f0, raw_cell="best_model, history = train_model_with_val_noteboo.." transformed_cell="best_model, history = train_model_with_val_noteboo.." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell://ssh-remote%2B7b22686f73744e616d65223a224e6575726f4c6162227d/mnt/e/fyassine/%2Bad-early-detection/notebooks/DenseGAAE_DANCER.ipynb#X12sdnNjb2RlLXJlbW90ZQ%3D%3D> result=None>,),kwargs {}:


KeyboardInterrupt: 